# Step 2 — Track mitolysosomes in the red channel

Takes the single-cell movies written by `step1-crop_image.ipynb` and, for each
cell, detects mitolysosome puncta in the red channel, links them into 3D
trajectories over time, and reduces each cell to six per-cell measurements.

Pipeline per cell:

1. **Detect** puncta with a difference-of-Gaussians filter (`stracking.DoGDetector`).
2. **Measure** the mean intensity of each punctum in a fixed-radius sphere.
3. **Link** detections across timepoints by nearest-neighbour Euclidean cost,
   tolerating short gaps where a punctum is missed.
4. **Filter** tracks by length to drop one-off detections and spurious long tracks.
5. **Reduce** to abundance and motility features, one row per measurement.

The green (mitochondrial) channel is used only as a per-frame normaliser, so
that abundance features are not confounded by how much mitochondrial mass a
given cell happens to contain.

### Features written to the results table

| # | Feature | Unit of observation |
|---|---------|---------------------|
| 1 | Mitolysosome mean intensity | one punctum |
| 2 | Mitolysosome count | one timepoint |
| 3 | Normalized mitolysosome intensity | one timepoint |
| 4 | Normalized mitolysosome count | one timepoint |
| 5 | Net mitolysosome displacement (µm) | one track |
| 6 | Average mitolysosome speed (µm/s) | one track |

In [ ]:
import os
from glob import glob

import matplotlib.pyplot as plt
import napari
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from skimage.filters import threshold_otsu
from tifffile import imread

from stracking.detectors import DoGDetector
from stracking.features import DisplacementFeature, DistanceFeature, LengthFeature
from stracking.filters import FeatureFilter
from stracking.linkers import EuclideanCost, SPLinker
from stracking.properties import IntensityProperty

## Configuration

In [ ]:
# Output of step 1, and where the results table / figures should be written.
ANALYZED_DIR = "/path/to/MOSAIC_Data/Analyzed_Data/Pham THP1"
RESULTS_DIR = "/path/to/MOSAIC_Data/Analyses_Results/Macrophage_Pham"

CONDITIONS = [
    "Control",
    "PGE2-activator",
    "PGE2-activator-EP4-inhibitor",
    "PGE2-activator-PKA-inhibitor",
]

# Acquisition geometry. Voxels are isotropic after deskewing.
PIXEL_SIZE_UM = 0.111
FRAME_INTERVAL_S = 11.3
N_FRAMES = 60

# Detection: sigma bounds bracket the expected punctum radius / 2.355.
# Lower the threshold to detect more (and dimmer) puncta.
DOG_PARAMS = dict(min_sigma=3.0, max_sigma=6.0, threshold=0.015)

# Radius (px) of the sphere over which each punctum's mean intensity is taken.
INTENSITY_RADIUS = 4

# Linking: MAX_LINK_COST is a squared distance in px^2; MAX_GAP is how many
# consecutive timepoints a punctum may go undetected without breaking its track.
MAX_LINK_COST = 800
MAX_GAP = 2

# Track length filter, in timepoints.
MIN_TRACK_LENGTH, MAX_TRACK_LENGTH = 4, 60

# Open a blocking napari window for every cell. Useful for inspecting the
# detections and tracks; must be False for an unattended batch run.
VISUALIZE = False

# Display range used by the napari preview only.
CONTRAST_LIMITS = [1500, 10000]

CONDITION_COLORS = {
    "Control": "cornflowerblue",
    "PGE2-activator": "orange",
    "PGE2-activator-EP4-inhibitor": "g",
    "PGE2-activator-PKA-inhibitor": "r",
}

## Helpers

In [ ]:
def load_movie(channel_dir):
    """Load `frame_*.tif` from one ROI channel folder as a (t, z, y, x) array.

    Frames are ordered by file creation time, which matches the order step 1
    wrote them in. Copying the folder with a tool that does not preserve
    timestamps can scramble this, so re-crop rather than copy.
    """
    frame_paths = glob(os.path.join(channel_dir, "frame*"))
    frame_paths.sort(key=os.path.getctime)
    return np.array([imread(path) for path in frame_paths])


def detect_and_track(movie):
    """Detect puncta, measure their intensity, and link them into filtered tracks."""
    spots = DoGDetector(**DOG_PARAMS).run(movie)
    spots.scale = (1, 1, 1)

    # Adds spots.properties["mean_intensity"], one value per detection.
    IntensityProperty(radius=INTENSITY_RADIUS).run(spots, movie)

    linker = SPLinker(cost=EuclideanCost(max_cost=MAX_LINK_COST), gap=MAX_GAP)
    tracks = linker.run(spots)
    tracks.scale = (1, 1, 1, 1)

    # Features are computed before filtering so that the filter can act on them.
    for feature in (LengthFeature, DistanceFeature, DisplacementFeature):
        tracks = feature().run(tracks)

    tracks = FeatureFilter(
        "length", min_val=MIN_TRACK_LENGTH, max_val=MAX_TRACK_LENGTH
    ).run(tracks)
    return spots, tracks


def totals_per_frame(spots, intensities, n_frames):
    """Number of puncta and summed punctum intensity at each timepoint."""
    times = spots.data[:, 0].astype(int)
    counts = np.array([np.sum(times == t) for t in range(n_frames)], dtype=float)
    intensity_sums = np.array(
        [intensities[times == t].sum() for t in range(n_frames)], dtype=float
    )
    return counts, intensity_sums


def mito_totals_per_frame(green_dir, n_frames):
    """Summed intensity and voxel count of the Otsu-thresholded mitochondrial signal."""
    intensity, voxels = [], []
    for t in range(n_frames):
        mito = imread(os.path.join(green_dir, f"frame_{t}.tif"))
        above = mito[mito > threshold_otsu(mito)]
        intensity.append(float(above.sum()))
        voxels.append(float(above.size))
    return np.array(intensity), np.array(voxels)


def feature_values(tracks, name):
    """Per-track feature values as an array, in track order."""
    return np.array(list(tracks.features[name].values()))


def show_in_napari(movie, spots, tracks):
    """Open a blocking 3D napari view of one cell with its detections and tracks."""
    viewer = napari.Viewer(ndisplay=3)
    viewer.add_image(movie, name="Mitolysosome", contrast_limits=CONTRAST_LIMITS)

    viewer.scale_bar.visible = True
    viewer.scale_bar.unit = "µm"
    viewer.scale_bar.color = "white"
    viewer.scale_bar.position = "bottom_right"
    viewer.scale_bar.font_size = 15

    viewer.add_points(
        spots.data, name="detections", size=4, shading="spherical", blending="additive"
    )
    viewer.add_tracks(tracks.data, name="tracks", tail_length=8, colormap="turbo")
    napari.run()

## Optional — inspect a single cell

Use this to sanity-check the crops and to tune `DOG_PARAMS` / `MAX_LINK_COST`
before running the batch. Skip straight to the next section for a batch run.

In [ ]:
all_red_dirs = sorted(glob(os.path.join(ANALYZED_DIR, "*", "roi_id_*", "red")))
print(len(all_red_dirs), "cells found")
for i, path in enumerate(all_red_dirs):
    print(i, os.path.relpath(path, ANALYZED_DIR))

In [ ]:
movie = load_movie(all_red_dirs[0])
print("(t, z, y, x) =", movie.shape)

spots, tracks = detect_and_track(movie)
print(len(spots.data), "detections ->", len(tracks.features["length"]), "tracks kept")

show_in_napari(movie, spots, tracks)

## Batch processing

Runs every cell of every condition and collects all six features into one long
(tidy) table: one row per measurement, so features with different units of
observation can live side by side.

In [ ]:
records = []


def record(condition, cell_id, feature, values):
    """Append one tidy block of rows to the results table."""
    records.append(
        pd.DataFrame(
            {
                "condition": condition,
                "cell_id": str(cell_id),
                "feature": feature,
                "value": np.ravel(values),
            }
        )
    )


for condition in CONDITIONS:
    red_dirs = sorted(glob(os.path.join(ANALYZED_DIR, condition, "roi_id_*", "red")))

    for cell_id, red_dir in enumerate(red_dirs):
        movie = load_movie(red_dir)
        spots, tracks = detect_and_track(movie)
        punctum_intensity = spots.properties["mean_intensity"]

        if VISUALIZE:
            show_in_napari(movie, spots, tracks)

        # --- abundance ---------------------------------------------------
        counts, intensity_sums = totals_per_frame(spots, punctum_intensity, N_FRAMES)
        green_dir = os.path.join(os.path.dirname(red_dir), "green")
        mito_intensity, mito_voxels = mito_totals_per_frame(green_dir, N_FRAMES)

        record(condition, cell_id, "Mitolysosome mean intensity", punctum_intensity)
        record(condition, cell_id, "Mitolysosome count", counts)
        record(
            condition,
            cell_id,
            "Normalized mitolysosome intensity",
            intensity_sums / mito_intensity,
        )
        record(
            condition, cell_id, "Normalized mitolysosome count", counts / mito_voxels
        )

        # --- motility ----------------------------------------------------
        length = feature_values(tracks, "length")
        distance = feature_values(tracks, "distance")
        displacement = feature_values(tracks, "displacement")
        duration_s = (length - 1) * FRAME_INTERVAL_S

        record(
            condition,
            cell_id,
            "Net mitolysosome displacement (µm)",
            displacement * PIXEL_SIZE_UM,
        )
        record(
            condition,
            cell_id,
            "Average mitolysosome speed (µm/s)",
            distance * PIXEL_SIZE_UM / duration_s,
        )

        print(f"Completed {condition} cell {cell_id}")

df = pd.concat(records, ignore_index=True)
df

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
df.to_csv(os.path.join(RESULTS_DIR, "all_features.csv"), index=False)

## Figures

One panel per feature. `cutoff` clips the long right tail of each distribution
so the panels stay readable; the dashed condition means are computed on the
**unclipped** values.

In [ ]:
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 14,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
})

# xticks=n forces n evenly spaced ticks where matplotlib's default is unhelpful.
PANELS = [
    dict(feature="Mitolysosome mean intensity", cutoff=20000, bins=50, kde=True),
    dict(feature="Mitolysosome count", cutoff=100, bins=50, kde=False, xticks=5),
    dict(feature="Normalized mitolysosome intensity", cutoff=0.0015, bins=50, kde=False, xticks=5),
    dict(feature="Normalized mitolysosome count", cutoff=0.0004, bins=50, kde=False, xticks=5),
    dict(feature="Net mitolysosome displacement (µm)", cutoff=11, bins=50, kde=True),
    dict(feature="Average mitolysosome speed (µm/s)", cutoff=0.27, bins=40, kde=True),
]

fig, axs = plt.subplots(len(PANELS), 1, figsize=(8, 20))

for ax, panel in zip(axs, PANELS):
    feature, cutoff = panel["feature"], panel["cutoff"]
    feature_df = df[df["feature"] == feature]

    sns.histplot(
        ax=ax,
        data=feature_df[feature_df["value"] < cutoff],
        x="value",
        hue="condition",
        stat="probability",
        common_norm=False,
        kde=panel["kde"],
        element="step",
        fill=True,
        bins=panel["bins"],
        binrange=(0, cutoff),
        alpha=0.2,
    )
    ax.set_xlabel(feature)
    ax.set_xlim(0, cutoff)
    if "xticks" in panel:
        ax.set_xticks(np.linspace(0, cutoff, panel["xticks"]))
    ax.get_legend().set_title(None)

    for condition, group in feature_df.groupby("condition"):
        ax.axvline(group["value"].mean(), color=CONDITION_COLORS[condition], linestyle="--")

fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "all_features_histograms.png"), dpi=300)

In [ ]:
# Single-panel box plot of normalized mitolysosome count.
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 14,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
})

cutoff = 0.00008
data = df[(df["feature"] == "Normalized mitolysosome count") & (df["value"] < cutoff)]

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
sns.boxplot(
    ax=ax,
    data=data,
    y="value",
    hue="condition",
    gap=0.5,
    whis=10,
    linewidth=1.5,
    boxprops=dict(alpha=0.8),
)
ax.set_ylabel("Normalized mitolysosome count")
ax.set_ylim(-0.05 * cutoff, cutoff * 1.1)
ax.set_xticks([])
ax.get_legend().set_title(None)
ax.legend(loc=9)

fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "normalized_count_boxplot.png"), dpi=300)

## Statistics

Two-sided Mann–Whitney U (distributions are non-normal and sample sizes differ
between conditions). Every condition is compared against `PGE2-activator`.

In [ ]:
REFERENCE = "PGE2-activator"

for feature in df["feature"].unique():
    feature_df = df[df["feature"] == feature]
    reference_values = feature_df[feature_df["condition"] == REFERENCE]["value"].values

    for condition in CONDITIONS:
        if condition == REFERENCE:
            continue
        values = feature_df[feature_df["condition"] == condition]["value"].values
        _, p_value = stats.mannwhitneyu(values, reference_values, alternative="two-sided")
        print(f"{feature}: {condition} vs {REFERENCE}  p = {p_value:.4g}")
    print()